# 02_02 · Preprocesado — CNC Mill Tool Wear


In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

CNC_DIR   = '../data/raw/archive'


## 3. CNC Mill — Carga y Agregación por experimento

### ¿Por qué agregar?
Cada experimento CNC es una **serie temporal de ~1.400 mediciones** de sensores.
Los modelos de clasificación estándar (RF, LGBM) no aceptan series de longitud variable.

**Estrategia:** resumir cada serie con estadísticos descriptivos por señal:
- `mean` → valor central / nivel de carga
- `std` → variabilidad / oscilaciones
- `max` → pico máximo (momentos de mayor esfuerzo)
- `min` → valor mínimo (vacíos de proceso)

Esto convierte 18 series × ~1.400 puntos en **18 vectores de ~184 features** —
un dataset clásico de clasificación tabular.

> **Limitación conocida:** la agregación pierde información temporal (tendencias, transitorios).
> Features temporales más sofisticadas (pendiente, FFT) podrían mejorar los resultados.


In [2]:
train_meta = pd.read_csv(f'{CNC_DIR}/train.csv').rename(columns={'No': 'experiment'})

dfs = []
for i in range(1, 19):
    df = pd.read_csv(f'{CNC_DIR}/experiment_{i:02d}.csv')
    df['experiment'] = i
    dfs.append(df)
cnc_raw = pd.concat(dfs, ignore_index=True)
cnc_raw = cnc_raw.merge(train_meta, on='experiment', how='left')

print('CNC raw shape:', cnc_raw.shape)


CNC raw shape: (25286, 55)


In [3]:
# Columnas de sensores (excluir columnas de control y metadata)
exclude = ['experiment', 'Machining_Process', 'M1_CURRENT_PROGRAM_NUMBER',
           'M1_sequence_number', 'material', 'feedrate', 'clamp_pressure',
           'tool_condition', 'machining_finalized', 'passed_visual_inspection']
sensor_cols = [c for c in cnc_raw.columns if c not in exclude]

# Agregar por experimento: media, std, max, min de cada señal
agg_dict = {c: ['mean', 'std', 'max', 'min'] for c in sensor_cols}
cnc_agg = cnc_raw.groupby('experiment').agg(agg_dict)
cnc_agg.columns = ['_'.join(c) for c in cnc_agg.columns]
cnc_agg = cnc_agg.reset_index()

# Añadir metadata de experimento
cnc_agg = cnc_agg.merge(
    train_meta[['experiment', 'material', 'feedrate', 'clamp_pressure', 'tool_condition']],
    on='experiment'
)

# Encoding y target
cnc_agg['material_enc'] = LabelEncoder().fit_transform(cnc_agg['material'])
cnc_agg['target'] = (cnc_agg['tool_condition'] == 'worn').astype(int)

print('CNC agregado shape:', cnc_agg.shape)
print(f'worn: {cnc_agg["target"].sum()} | unworn: {(cnc_agg["target"]==0).sum()}')


CNC agregado shape: (18, 187)
worn: 10 | unworn: 8


## 4. CNC — Split

Con **18 muestras** en total (10 worn, 8 unworn), cualquier split fijo 80/20 deja
solo 3-4 muestras en test — demasiado poco para estimar métricas fiables.

**Solución:** LeaveOneOut Cross-Validation (LOO-CV):
- En cada iteración, entrenamos con 17 muestras y evaluamos con 1
- Repetimos 18 veces → cada muestra actúa como test exactamente una vez
- Las métricas promediadas sobre las 18 iteraciones son más estables que un único split

También guardamos un split 80/20 para comparaciones rápidas, pero LOO-CV
es la referencia de evaluación para CNC.


In [4]:
from sklearn.model_selection import LeaveOneOut

feature_cols_cnc = [c for c in cnc_agg.columns
                    if c not in ['experiment', 'material', 'tool_condition', 'target']]

X_cnc = cnc_agg[feature_cols_cnc].fillna(0)
y_cnc = cnc_agg['target']

scaler_cnc = StandardScaler()
X_cnc_scaled = scaler_cnc.fit_transform(X_cnc)

# Split 80/20 para métricas rápidas
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cnc_scaled, y_cnc, test_size=0.2, random_state=42)

print(f'CNC features: {X_cnc.shape[1]}')
print(f'CNC Train: {X_train_c.shape} | Test: {X_test_c.shape}')
print('LOO-CV se aplicará en el notebook 03 (evaluación más robusta con 18 muestras).')


CNC features: 183
CNC Train: (14, 183) | Test: (4, 183)
LOO-CV se aplicará en el notebook 03 (evaluación más robusta con 18 muestras).


## 5. Guardar datos procesados CNC

Guardamos `cnc_aggregated.csv` y actualizamos `splits.pkl` con las claves `cnc` y `cnc_loo`.


In [5]:
import pickle, os
os.makedirs('../data/processed', exist_ok=True)

pkl_path = '../data/processed/splits.pkl'
splits = {}
if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        splits = pickle.load(f)

splits['cnc'] = (X_train_c, X_test_c, y_train_c, y_test_c)
splits['cnc_loo'] = (X_cnc_scaled, y_cnc)
with open(pkl_path, 'wb') as f:
    pickle.dump(splits, f)

cnc_agg.to_csv('../data/processed/cnc_aggregated.csv', index=False)
print('CNC guardado OK.')
print(f'Train: {X_train_c.shape} | Test: {X_test_c.shape}')
print(f'Splits disponibles: {list(splits.keys())}')


CNC guardado OK.
Train: (14, 183) | Test: (4, 183)
Splits disponibles: ['ai4i', 'cnc', 'cnc_loo']
